# Pipeline de traitement des données OpenFoodFacts pour l'Éco-Score (Polars)

Ce notebook présente une méthode de lecture et de traitement **par lot (batch)** du gros fichier CSV de OpenFoodFacts (~12 Go) spécifiquement pour l'Éco-Score.

Il produit **deux fichiers CSV** :
1. `produits_sans_ecoscore.csv` – Produits français **SANS** Éco-Score, mais avec suffisamment de données renseignées.
2. `produits_avec_ecoscore.csv` – Produits français **AVEC** Éco-Score établi et les features textuelles/nutritionnelles importantes présentes.

In [ ]:
import polars as pl
import os

## Configuration

In [ ]:
# Fichier source situé à la racine du projet
FILE_PATH = "../data_brut.csv"
BATCH_SIZE = 100_000

# Features importantes pour l'Éco-Score
FEATURES_ECO = [
    'categories', 
    'labels', 
    'packaging', 
    'origins', 
    'ingredients_text', 
    'ingredients_analysis_tags'
]

COLONNES_NUTRITIONNELLES = '^.*_100g$'

if not os.path.exists(FILE_PATH):
    print(f"Attention : Fichier introuvable : {FILE_PATH}")
else:
    print(f"Fichier source : {FILE_PATH}")

## Pipeline 1 — Produits français SANS Éco-Score

Critères de sélection :
- Vendu en France (`countries_en` contient `france`)
- `environmental_score_score` est **null** (pas encore noté)
- Au moins **15 colonnes non nulles**

In [ ]:
destination_sans = "produits_sans_ecoscore.csv"

reader = pl.read_csv_batched(
    FILE_PATH,
    separator='\t',
    has_header=True,
    ignore_errors=True,
    infer_schema_length=10000,
    low_memory=True,
    quote_char=None,
    batch_size=BATCH_SIZE
)

batches_sans = []
i = 1
print("Début du Pipeline 1 (sans Éco-Score)...\n")

while True:
    batches = reader.next_batches(1)
    if not batches:
        break
    df_batch = batches[0]

    df_batch = (
        df_batch
        .filter(pl.col('countries_en').str.to_lowercase().str.contains('france'))
        .select([
            pl.col('code'),
            pl.col('product_name'),
            pl.col('nova_group'),
            pl.col('additives_n'),
            pl.col(COLONNES_NUTRITIONNELLES),
            pl.col('nutriscore_score'),
            pl.col('nutriscore_grade'),
            pl.col('environmental_score_score'),
            pl.col('environmental_score_grade'),
            # On ajoute les features textuelles
            pl.col('categories'),
            pl.col('labels'),
            pl.col('packaging'),
            pl.col('origins'),
            pl.col('ingredients_text'),
            pl.col('ingredients_analysis_tags')
        ])
        .filter(pl.col('environmental_score_score').is_null())
        .with_columns(
            pl.sum_horizontal(pl.all().is_not_null()).alias('donnees_fournies_count')
        )
        .filter(pl.col('donnees_fournies_count') >= 15)
    )

    batches_sans.append(df_batch)
    print(f"  Échantillon {i} : {len(df_batch)} produits conservés")
    i += 1

if batches_sans:
    df_sans = pl.concat(batches_sans).sort('donnees_fournies_count', descending=True)
    df_sans.write_csv(destination_sans)
    print(f"\nFichier sauvegardé : {destination_sans}")
    print(f"Dimensions : {df_sans.shape}")
else:
    print("Aucun produit sans éco-score n'a été trouvé.")

## Pipeline 2 — Produits français AVEC Éco-Score (données d'entraînement)

Critères de sélection :
- Vendu en France (`countries_en` contient `france`)
- `environmental_score_grade` est **une lettre valide** (a / b / c / d / e)
- On requiert la présence d'au moins la catégorie et la liste d'ingrédients pour avoir des données utiles à l'entraînement.

In [ ]:
destination_avec = "produits_avec_ecoscore.csv"

GRADES_VALIDES = ['a', 'b', 'c', 'd', 'e']
batches_avec = []
print("Début du Pipeline 2 (avec Éco-Score)...\n")

colonnes_base = [
    'countries_en',
    'code',
    'product_name',
    'nova_group',
    'additives_n',
    'nutriscore_score',
    'nutriscore_grade',
    'environmental_score_score',
    'environmental_score_grade',
    'categories', 'labels', 'packaging', 'origins', 'ingredients_text', 'ingredients_analysis_tags'
]

# ---------- Pass 1 : compter les classes disponibles ----------
reader2_count = pl.read_csv_batched(
    FILE_PATH,
    separator='\t',
    has_header=True,
    ignore_errors=True,
    infer_schema_length=10000,
    low_memory=True,
    quote_char=None,
    batch_size=BATCH_SIZE
)

compteurs = {g: 0 for g in GRADES_VALIDES}

while True:
    batches_count = reader2_count.next_batches(1)
    if not batches_count:
        break

    df_count = batches_count[0]
    # Vérifier que les colonnes nécessaires sont là
    colonnes_requises = ['countries_en', 'environmental_score_grade', 'categories', 'ingredients_text']
    if any(c not in df_count.columns for c in colonnes_requises):
        continue

    dist = (
        df_count
        .filter(pl.col('countries_en').str.to_lowercase().str.contains('france'))
        .with_columns(pl.col('environmental_score_grade').str.to_lowercase().alias('environmental_score_grade'))
        .filter(pl.col('environmental_score_grade').is_in(GRADES_VALIDES))
        # On exige au minimum la catégorie et les ingrédients pour l'apprentissage
        .filter(pl.col('categories').is_not_null() & pl.col('ingredients_text').is_not_null())
        .group_by('environmental_score_grade')
        .len()
    )

    for row in dist.iter_rows(named=True):
        compteurs[row['environmental_score_grade']] += row['len']

TARGET_PAR_GRADE = min(compteurs.values()) if all(v > 0 for v in compteurs.values()) else 0
print(f"Comptes initiaux par grade : {compteurs}")
print(f"Cible équilibrée par grade : {TARGET_PAR_GRADE}\n")

# ---------- Pass 2 : extraire de façon équilibrée ----------
reader2 = pl.read_csv_batched(
    FILE_PATH,
    separator='\t',
    has_header=True,
    ignore_errors=True,
    infer_schema_length=10000,
    low_memory=True,
    quote_char=None,
    batch_size=BATCH_SIZE
)

quota_restant = {g: TARGET_PAR_GRADE for g in GRADES_VALIDES}

def equilibrer_batch(df: pl.DataFrame) -> pl.DataFrame:
    if TARGET_PAR_GRADE == 0:
        return df.head(0)

    df = (
        df
        .with_columns(pl.col('environmental_score_grade').str.to_lowercase().alias('environmental_score_grade'))
        .filter(pl.col('environmental_score_grade').is_in(GRADES_VALIDES))
        .filter(pl.col('categories').is_not_null() & pl.col('ingredients_text').is_not_null())
    )

    morceaux = []
    for g in GRADES_VALIDES:
        reste = quota_restant[g]
        if reste <= 0:
            continue

        part = df.filter(pl.col('environmental_score_grade') == g)
        if len(part) == 0:
            continue

        n = min(len(part), reste)
        part = part.sample(n=n, seed=42, shuffle=True) if len(part) > n else part.head(n)
        quota_restant[g] -= n
        morceaux.append(part)

    return pl.concat(morceaux) if morceaux else df.head(0)

j = 1
while True:
    if TARGET_PAR_GRADE > 0 and all(v == 0 for v in quota_restant.values()):
        print("Quotas atteints pour tous les grades, arrêt anticipé.")
        break

    batches = reader2.next_batches(1)
    if not batches:
        break

    df_batch = batches[0]

    colonnes_manquantes = [c for c in colonnes_base if c not in df_batch.columns]
    if colonnes_manquantes:
        print(f"  Échantillon {j} ignoré (colonnes manquantes: {colonnes_manquantes})")
        j += 1
        continue

    df_batch = (
        df_batch
        .filter(pl.col('countries_en').str.to_lowercase().str.contains('france'))
        .select([
            pl.col('code'),
            pl.col('product_name'),
            pl.col('nova_group'),
            pl.col('additives_n'),
            pl.col(COLONNES_NUTRITIONNELLES),
            pl.col('nutriscore_score'),
            pl.col('nutriscore_grade'),
            pl.col('environmental_score_score'),
            pl.col('environmental_score_grade'),
            pl.col('categories'),
            pl.col('labels'),
            pl.col('packaging'),
            pl.col('origins'),
            pl.col('ingredients_text'),
            pl.col('ingredients_analysis_tags')
        ])
        .pipe(equilibrer_batch)
    )

    batches_avec.append(df_batch)
    print(f"  Échantillon {j} : {len(df_batch)} produits conservés")
    j += 1

if batches_avec:
    df_avec = pl.concat(batches_avec)
    df_avec.write_csv(destination_avec)
    print(f"\nFichier sauvegardé : {destination_avec}")
    print(f"Dimensions : {df_avec.shape}")
    print("\nDistribution des grades :")
    if "environmental_score_grade" in df_avec.columns:
        print(df_avec['environmental_score_grade'].value_counts())
else:
    print("Aucun produit avec éco-score n'a été trouvé.")